In [1]:
import idaes
from pyomo.environ import ConcreteModel, SolverFactory
import idaes.logger as idaeslog

# Check version
print("IDAES version:", idaes.__version__)

# Check IPOPT solver
solver = SolverFactory("ipopt")
print("IPOPT available:", solver.available())

IDAES version: 2.11.0
IPOPT available: True


In [2]:
# Step 0 — Verify all MAPIRL-DT imports work

import idaes
print('IDAES:', idaes.__version__)  # must say 2.11.0

from idaes.models_extra.power_generation.unit_models.soec_design import SoecDesign
from idaes.models_extra.power_generation.properties.natural_gas_PR import get_prop, EosType

print('All imports OK')


IDAES: 2.11.0
All imports OK


In [3]:
from pyomo.environ import ConcreteModel, SolverFactory, value, TransformationFactory
from idaes.core import FlowsheetBlock
from idaes.models_extra.power_generation.unit_models.soec_design import SoecDesign
from idaes.models_extra.power_generation.properties.natural_gas_PR import get_prop, EosType
from idaes.models.properties.modular_properties.base.generic_property import GenericParameterBlock

m = ConcreteModel()
m.fs = FlowsheetBlock(dynamic=False)
print("Flowsheet built OK")

Flowsheet built OK


In [4]:
# Cathode (hydrogen side): H2O + H2
m.fs.prop_h2 = GenericParameterBlock(
    **get_prop({"H2O", "H2"}, phases={"Vap"}, eos=EosType.IDEAL)
)

# Anode (oxygen side): O2 + H2O sweep gas
m.fs.prop_o2 = GenericParameterBlock(
    **get_prop({"O2", "H2O"}, phases={"Vap"}, eos=EosType.IDEAL)
)
print("Property packages built OK")

Property packages built OK


In [5]:
m.fs.soec = SoecDesign(
    oxygen_side_property_package=m.fs.prop_o2,
    hydrogen_side_property_package=m.fs.prop_h2,
    has_heat_transfer=True,
)
TransformationFactory("network.expand_arcs").apply_to(m)
print("SoecDesign configured OK")

SoecDesign configured OK


In [6]:
# Hydrogen side inlet (cathode)
m.fs.soec.hydrogen_side.inlet.temperature[0].fix(1073.15)  # 800°C
m.fs.soec.hydrogen_side.inlet.pressure[0].fix(1.5e5)
m.fs.soec.hydrogen_side.inlet.flow_mol[0].fix(0.10)
m.fs.soec.hydrogen_side.inlet.mole_frac_comp[0, "H2O"].fix(0.90)
m.fs.soec.hydrogen_side.inlet.mole_frac_comp[0, "H2"].fix(0.10)

# Oxygen side inlet (anode)
m.fs.soec.oxygen_side.inlet.temperature[0].fix(1073.15)
m.fs.soec.oxygen_side.inlet.pressure[0].fix(1.5e5)
m.fs.soec.oxygen_side.inlet.flow_mol[0].fix(0.10)
m.fs.soec.oxygen_side.inlet.mole_frac_comp[0, "O2"].fix(0.21)
m.fs.soec.oxygen_side.inlet.mole_frac_comp[0, "H2O"].fix(0.79)

# Design variable
m.fs.soec.water_utilization[0].fix(0.80)

# ⚠️ CRITICAL CHECK — must print 0 before solving
from idaes.core.util.model_statistics import degrees_of_freedom
print("Degrees of freedom:", degrees_of_freedom(m))  # Must be 0

AttributeError: '_ScalarSoecDesign' object has no attribute 'hydrogen_side'

In [7]:
# Run this BEFORE fixing any variables
from idaes.models_extra.power_generation.unit_models.soec_design import SoecDesign
import inspect, os

# Find where the file actually lives
src_path = inspect.getfile(SoecDesign)
print("Source file:", src_path)

# Print all component names on the built model
print("\n--- All soec attributes ---")
for name in m.fs.soec.component_map():
    print(name)

Source file: C:\Users\LOGIN\miniconda3\envs\idaes-env\lib\site-packages\idaes\models_extra\power_generation\unit_models\soec_design.py

--- All soec attributes ---
electrolysis_prop_params
electrolysis_rxn_params
h2_inlet_translator
electrolysis_reactor
o2_separator
h2_outlet_translator
o2_translator
o2_mixer
sweep_heater
strm1
strm2
strm3
strm4
strm5
strm6
strm1_expanded
strm2_expanded
strm3_expanded
strm4_expanded
strm5_expanded
strm6_expanded
current
water_utilization
cell_potential
heat
hydrogen_side_outlet_temperature
oxygen_side_outlet_temperature
water_utilization_eqn
current_expr
current_eqn
delta_enth
cell_potential_eqn
hydrogen_side_inlet
_flow_mol_hydrogen_side_inlet_ref
_mole_frac_comp_hydrogen_side_inlet_ref
_temperature_hydrogen_side_inlet_ref
_pressure_hydrogen_side_inlet_ref
oxygen_side_inlet
_flow_mol_oxygen_side_inlet_ref
_mole_frac_comp_oxygen_side_inlet_ref
_temperature_oxygen_side_inlet_ref
_pressure_oxygen_side_inlet_ref
hydrogen_side_outlet
_flow_mol_hydrogen_sid

In [8]:
# Find all inlet/outlet ports
print("\n--- Ports only ---")
from pyomo.network import Port
for name, comp in m.fs.soec.component_map():
    if isinstance(comp, Port):
        print(f"PORT: {name}")
        for var in comp.vars:
            print(f"  .{var}")


--- Ports only ---


ValueError: too many values to unpack (expected 2)

In [9]:
# Step 4 — Fix conditions using CORRECT port names (confirmed from component_map)

# Hydrogen side inlet (cathode)
m.fs.soec.hydrogen_side_inlet.flow_mol[0].fix(0.10)
m.fs.soec.hydrogen_side_inlet.mole_frac_comp[0, "H2O"].fix(0.90)
m.fs.soec.hydrogen_side_inlet.mole_frac_comp[0, "H2"].fix(0.10)
m.fs.soec.hydrogen_side_inlet.temperature[0].fix(1073.15)   # 800°C
m.fs.soec.hydrogen_side_inlet.pressure[0].fix(1.5e5)        # 1.5 bar

# Oxygen side inlet (anode)
m.fs.soec.oxygen_side_inlet.flow_mol[0].fix(0.10)
m.fs.soec.oxygen_side_inlet.mole_frac_comp[0, "O2"].fix(0.21)
m.fs.soec.oxygen_side_inlet.mole_frac_comp[0, "H2O"].fix(0.79)
m.fs.soec.oxygen_side_inlet.temperature[0].fix(1073.15)
m.fs.soec.oxygen_side_inlet.pressure[0].fix(1.5e5)

# Design variable — controls effective current density
m.fs.soec.water_utilization[0].fix(0.80)

# ⚠️ MUST print 0 before solving
from idaes.core.util.model_statistics import degrees_of_freedom
print("Degrees of freedom:", degrees_of_freedom(m))

Degrees of freedom: 3


In [10]:
# Diagnose which variables are free (unfixed)
from idaes.core.util.model_statistics import (
    degrees_of_freedom,
    unfixed_variables_in_activated_equalities
)

print("DOF:", degrees_of_freedom(m))
print("\n--- Unfixed variables ---")
for v in unfixed_variables_in_activated_equalities(m):
    print(v)

ImportError: cannot import name 'unfixed_variables_in_activated_equalities' from 'idaes.core.util.model_statistics' (C:\Users\LOGIN\miniconda3\envs\idaes-env\lib\site-packages\idaes\core\util\model_statistics.py)

In [11]:
# Find the 3 unfixed variables using Pyomo directly
from pyomo.environ import value
import pyomo.environ as pyo

print("--- Unfixed active variables ---")
count = 0
for v in m.component_data_objects(pyo.Var, active=True, descend_into=True):
    if not v.fixed and not v.stale:
        print(v, "=", v.value)
        count += 1
print(f"\nTotal unfixed: {count}")

--- Unfixed active variables ---
fs.soec.current[0.0] = 0
fs.soec.cell_potential[0.0] = 1.3
fs.soec.heat[0.0] = 0
fs.soec.electrolysis_reactor.control_volume.properties_out[0.0].temperature = 500
fs.soec.sweep_heater.control_volume.properties_out[0.0].temperature = 500
fs.soec.h2_outlet_translator.properties_out[0.0].flow_mol = 8000
fs.soec.h2_outlet_translator.properties_out[0.0].mole_frac_comp[H2O] = 0.5
fs.soec.h2_outlet_translator.properties_out[0.0].mole_frac_comp[H2] = 0.5
fs.soec.h2_outlet_translator.properties_out[0.0].temperature = 500
fs.soec.h2_outlet_translator.properties_out[0.0].pressure = 130000.0
fs.soec.sweep_heater.control_volume.properties_out[0.0].flow_mol = 8000
fs.soec.sweep_heater.control_volume.properties_out[0.0].mole_frac_comp[H2O] = 0.5
fs.soec.sweep_heater.control_volume.properties_out[0.0].mole_frac_comp[O2] = 0.5
fs.soec.sweep_heater.control_volume.properties_out[0.0].pressure = 130000.0
fs.soec.h2_inlet_translator.properties_out[0.0].flow_mol = 8000
fs.so

In [12]:
# Focused: only check soec component
print("--- Unfixed vars in m.fs.soec ---")
for v in m.fs.soec.component_data_objects(pyo.Var, active=True, descend_into=True):
    if not v.fixed:
        print(v)

--- Unfixed vars in m.fs.soec ---
fs.soec.current[0.0]
fs.soec.cell_potential[0.0]
fs.soec.heat[0.0]
fs.soec.electrolysis_reactor.control_volume.properties_out[0.0].temperature
fs.soec.sweep_heater.control_volume.properties_out[0.0].temperature
fs.soec.h2_outlet_translator.properties_out[0.0].flow_mol
fs.soec.h2_outlet_translator.properties_out[0.0].mole_frac_comp[H2O]
fs.soec.h2_outlet_translator.properties_out[0.0].mole_frac_comp[H2]
fs.soec.h2_outlet_translator.properties_out[0.0].temperature
fs.soec.h2_outlet_translator.properties_out[0.0].pressure
fs.soec.sweep_heater.control_volume.properties_out[0.0].flow_mol
fs.soec.sweep_heater.control_volume.properties_out[0.0].mole_frac_comp[H2O]
fs.soec.sweep_heater.control_volume.properties_out[0.0].mole_frac_comp[O2]
fs.soec.sweep_heater.control_volume.properties_out[0.0].pressure
fs.soec.h2_inlet_translator.properties_out[0.0].flow_mol
fs.soec.h2_inlet_translator.properties_out[0.0].mole_frac_comp[H2O]
fs.soec.h2_inlet_translator.propert

In [13]:
# Fix the 3 remaining top-level DOFs
# Option A: Fix cell_potential and heat (let current be computed)
m.fs.soec.cell_potential[0].fix(1.28)   # V — mid-range operating point
m.fs.soec.heat[0].fix(0.0)              # W — adiabatic assumption

# Leave water_utilization fixed at 0.80 (already done)
# current[0] will be computed by the solver

from idaes.core.util.model_statistics import degrees_of_freedom
print("Degrees of freedom:", degrees_of_freedom(m))  # Must print 0

Degrees of freedom: 1


In [14]:
# Find the last remaining unfixed top-level variable
print("--- Remaining top-level free vars ---")
for name in ["current", "cell_potential", "heat", "water_utilization", "scaling_factor"]:
    try:
        v = getattr(m.fs.soec, name)[0]
        print(f"{name}: fixed={v.fixed}, value={v.value}")
    except:
        pass

--- Remaining top-level free vars ---
current: fixed=False, value=0
cell_potential: fixed=True, value=1.28
heat: fixed=True, value=0.0
water_utilization: fixed=True, value=0.8


In [15]:
# Fix current — this closes the last DOF
# cell_potential_eqn will be satisfied by the solver
# Unfix cell_potential so it's computed, fix current instead
m.fs.soec.cell_potential[0].unfix()
m.fs.soec.current[0].fix(50000)  # 50,000 A starting point

from idaes.core.util.model_statistics import degrees_of_freedom
print("DOF:", degrees_of_freedom(m))  # Must be 0

DOF: 1


In [16]:
# Fix current to match your target operating point
# For 1.61 A/cm² with 900 cells × 360 cm²:
# current = 1.61 × 900 × 360 = 521,640 A
# Start conservative — use a lower value first
m.fs.soec.current[0].fix(50000)   # 50 kA starting point

from idaes.core.util.model_statistics import degrees_of_freedom
print("Degrees of freedom:", degrees_of_freedom(m))  # Must print 0

Degrees of freedom: 1


In [17]:
# Precisely identify the ONE remaining free variable
# by checking all top-level soec vars for fixed status
print("=== All top-level SOEC scalar/indexed vars ===")
import pyomo.environ as pyo

for comp in m.fs.soec.component_objects(pyo.Var, active=True, descend_into=False):
    for idx, v in comp.items():
        if not v.fixed:
            print(f"UNFIXED: {comp.name}[{idx}] = {v.value}")

print("\n--- Also checking scaling_factor ---")
try:
    sf = m.fs.soec.scaling_factor
    for k, v in sf.items():
        print(f"  scaling_factor[{k}] = {v}")
except:
    print("  no scaling_factor found")

=== All top-level SOEC scalar/indexed vars ===
UNFIXED: fs.soec.cell_potential[0.0] = 1.28
UNFIXED: fs.soec.hydrogen_side_outlet_temperature[0.0] = 500
UNFIXED: fs.soec.oxygen_side_outlet_temperature[0.0] = 500
UNFIXED: fs.soec._flow_mol_hydrogen_side_outlet_ref[0.0] = 8000
UNFIXED: fs.soec._mole_frac_comp_hydrogen_side_outlet_ref[(0.0, 'H2O')] = 0.5
UNFIXED: fs.soec._mole_frac_comp_hydrogen_side_outlet_ref[(0.0, 'H2')] = 0.5
UNFIXED: fs.soec._temperature_hydrogen_side_outlet_ref[0.0] = 500
UNFIXED: fs.soec._pressure_hydrogen_side_outlet_ref[0.0] = 130000.0
UNFIXED: fs.soec._flow_mol_oxygen_side_outlet_ref[0.0] = 8000
UNFIXED: fs.soec._mole_frac_comp_oxygen_side_outlet_ref[(0.0, 'H2O')] = 0.5
UNFIXED: fs.soec._mole_frac_comp_oxygen_side_outlet_ref[(0.0, 'O2')] = 0.5
UNFIXED: fs.soec._temperature_oxygen_side_outlet_ref[0.0] = 500
UNFIXED: fs.soec._pressure_oxygen_side_outlet_ref[0.0] = 130000.0

--- Also checking scaling_factor ---
  scaling_factor[fs.soec.heat] = 1e-05
  scaling_factor

In [18]:
# Clean reset — establish correct DOF=0 state
# Strategy: fix cell_potential as design input, unfix current (it's computed)

m.fs.soec.current[0].unfix()          # current is an OUTPUT — let solver compute it
m.fs.soec.cell_potential[0].fix(1.28) # cell voltage is the DESIGN INPUT

from idaes.core.util.model_statistics import degrees_of_freedom
print("DOF:", degrees_of_freedom(m))  # Must be 0

DOF: 1


In [19]:
# Find the ONE variable causing DOF=1 by checking sub-units
import pyomo.environ as pyo
from idaes.core.util.model_statistics import degrees_of_freedom

# Check DOF of each sub-unit individually
subunits = [
    "electrolysis_reactor",
    "sweep_heater", 
    "o2_separator",
    "o2_mixer",
    "h2_inlet_translator",
    "h2_outlet_translator",
    "o2_translator"
]

for name in subunits:
    try:
        unit = getattr(m.fs.soec, name)
        dof = degrees_of_freedom(unit)
        if dof != 0:
            print(f"*** DOF={dof} in: m.fs.soec.{name}")
        else:
            print(f"    DOF=0 OK: {name}")
    except Exception as e:
        print(f"    Could not check {name}: {e}")

*** DOF=8 in: m.fs.soec.electrolysis_reactor
*** DOF=6 in: m.fs.soec.sweep_heater
*** DOF=6 in: m.fs.soec.o2_separator
*** DOF=4 in: m.fs.soec.o2_mixer
    DOF=0 OK: h2_inlet_translator
*** DOF=6 in: m.fs.soec.h2_outlet_translator
*** DOF=6 in: m.fs.soec.o2_translator


In [20]:
# The sweep_heater outlet temperature is the missing specification
# It controls the oxygen-side inlet temperature to the electrolysis reactor
m.fs.soec.sweep_heater.outlet.temperature[0].fix(1073.15)  # 800°C = same as inlet

from idaes.core.util.model_statistics import degrees_of_freedom
print("DOF:", degrees_of_freedom(m))

DOF: 0


In [21]:
from idaes.core.solvers import get_solver
from pyomo.environ import value

solver = get_solver()

try:
    m.fs.soec.initialize(outlvl=0)
except Exception as e:
    print(f"Init note: {e}")

result = solver.solve(m, tee=False)

print("\n=== SOLVE RESULT ===")
print("Status:   ", result.solver.termination_condition)
print("V_cell:   ", round(value(m.fs.soec.cell_potential[0]), 4), "V")
print("Current:  ", round(value(m.fs.soec.current[0]), 1), "A")
print("T_h2_out: ", round(value(m.fs.soec.hydrogen_side_outlet_temperature[0]), 2), "K")
print("T_o2_out: ", round(value(m.fs.soec.oxygen_side_outlet_temperature[0]), 2), "K")
dT = (value(m.fs.soec.hydrogen_side_outlet_temperature[0]) -
      value(m.fs.soec.oxygen_side_outlet_temperature[0]))
print("dT:       ", round(dT, 2), "K")
j = value(m.fs.soec.current[0]) / (900 * 360)
print("j:        ", round(j, 4), "A/cm²")
print("Safe:     ", "YES" if (0.9 <= value(m.fs.soec.cell_potential[0]) <= 1.3 
                               and abs(dT) <= 20) else "NO")

2026-03-28 03:17:25 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Starting initialization
2026-03-28 03:17:30 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Property initialization: optimal - Optimal Solution Found.
2026-03-28 03:17:30 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Starting initialization
2026-03-28 03:17:30 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property initialization: optimal - Optimal Solution Found.
2026-03-28 03:17:30 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property package initialization: optimal - Optimal Solution Found.
2026-03-28 03:17:30 [INFO] idaes.init.fs.soec.h2_inlet_translator: Initialization Complete optimal - Optimal Solution Found.
2026-03-28 03:17:30 [INFO] idaes.init.fs.soec.electrolysis_reactor.control_volume.properties_in: Starting initialization
that are not Var, Constraint, Objective, or the model.  Skipping.
2026-03-28 03:17:30 [INFO] idaes.init.fs.soec.e

In [22]:
# Cell — Checkpoint: save the working fixed-point state
import pickle
from pyomo.environ import value

checkpoint = {
    "description": "Single-point solve validated. DOF=0, Status=optimal.",
    "fixed_inputs": {
        "hydrogen_side_inlet.flow_mol": 0.10,
        "hydrogen_side_inlet.temperature": 1073.15,
        "hydrogen_side_inlet.pressure": 1.5e5,
        "hydrogen_side_inlet.mole_frac_H2O": 0.90,
        "hydrogen_side_inlet.mole_frac_H2": 0.10,
        "oxygen_side_inlet.flow_mol": 0.10,
        "oxygen_side_inlet.temperature": 1073.15,
        "oxygen_side_inlet.pressure": 1.5e5,
        "oxygen_side_inlet.mole_frac_O2": 0.21,
        "oxygen_side_inlet.mole_frac_H2O": 0.79,
        "water_utilization": 0.80,
        "cell_potential": 1.28,
        "heat": 0.0,
        "sweep_heater_outlet_temperature": 1073.15,
    },
    "validated_outputs": {
        "V_cell_V":    round(value(m.fs.soec.cell_potential[0]), 4),
        "current_A":   round(value(m.fs.soec.current[0]), 1),
        "T_h2_out_K":  round(value(m.fs.soec.hydrogen_side_outlet_temperature[0]), 2),
        "T_o2_out_K":  round(value(m.fs.soec.oxygen_side_outlet_temperature[0]), 2),
        "dT_K":        round(value(m.fs.soec.hydrogen_side_outlet_temperature[0]) -
                             value(m.fs.soec.oxygen_side_outlet_temperature[0]), 2),
        "j_Acm2":      round(value(m.fs.soec.current[0]) / (900 * 360), 6),
    }
}

with open("dt_checkpoint_single_solve.pkl", "wb") as f:
    pickle.dump(checkpoint, f)

print("=== CHECKPOINT SAVED ===")
for k, v in checkpoint["validated_outputs"].items():
    print(f"  {k}: {v}")
print("\nSafe to proceed to Step 5 sweep.")

=== CHECKPOINT SAVED ===
  V_cell_V: 1.28
  current_A: 13893.9
  T_h2_out_K: 1043.89
  T_o2_out_K: 1073.15
  dT_K: -29.26
  j_Acm2: 0.042882

Safe to proceed to Step 5 sweep.


In [23]:
import pandas as pd
from pyomo.environ import value

results = []
V_targets = [0.95, 1.00, 1.05, 1.10, 1.15, 1.20, 1.25, 1.28]

for v_target in V_targets:
    m.fs.soec.cell_potential[0].fix(v_target)
    
    try:
        m.fs.soec.initialize(outlvl=0)
    except:
        pass
    
    res = solver.solve(m, tee=False)
    
    if str(res.solver.termination_condition) == "optimal":
        v_cell = value(m.fs.soec.cell_potential[0])
        T_h2   = value(m.fs.soec.hydrogen_side_outlet_temperature[0])
        T_o2   = value(m.fs.soec.oxygen_side_outlet_temperature[0])
        dT     = T_h2 - T_o2
        current = value(m.fs.soec.current[0])
        j      = current / (900 * 360)
        safe   = int(0.9 <= v_cell <= 1.3 and abs(dT) <= 20)
        
        results.append({
            "cell_potential_V": round(v_cell, 4),
            "current_A":        round(current, 2),
            "j_Acm2":           round(j, 6),
            "T_h2_out_K":       round(T_h2, 2),
            "T_o2_out_K":       round(T_o2, 2),
            "dT_K":             round(dT, 2),
            "safe":             safe
        })
        print(f"V={v_cell:.2f}V  j={j:.4f}  dT={dT:.1f}K  safe={safe}")
    else:
        print(f"WARNING: failed at V={v_target}")

df = pd.DataFrame(results)
df.to_csv("pinn_training_data.csv", index=False)
print("\n", df.to_string())

2026-03-28 03:17:32 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Starting initialization
2026-03-28 03:17:32 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Property initialization: optimal - Optimal Solution Found.
2026-03-28 03:17:32 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Starting initialization
2026-03-28 03:17:33 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property initialization: optimal - Optimal Solution Found.
2026-03-28 03:17:33 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property package initialization: optimal - Optimal Solution Found.
2026-03-28 03:17:33 [INFO] idaes.init.fs.soec.h2_inlet_translator: Initialization Complete optimal - Optimal Solution Found.
2026-03-28 03:17:33 [INFO] idaes.init.fs.soec.electrolysis_reactor.control_volume.properties_in: Starting initialization
that are not Var, Constraint, Objective, or the model.  Skipping.
2026-03-28 03:17:33 [INFO] idaes.init.fs.soec.e

In [24]:
# ============================================================
# MAPIRL-DT FULL REBUILD — run this single cell to restore state
# ============================================================
from pyomo.environ import ConcreteModel, SolverFactory, value, TransformationFactory
from idaes.core import FlowsheetBlock
from idaes.models_extra.power_generation.unit_models.soec_design import SoecDesign
from idaes.models_extra.power_generation.properties.natural_gas_PR import get_prop, EosType
from idaes.models.properties.modular_properties.base.generic_property import GenericParameterBlock
from idaes.core.util.model_statistics import degrees_of_freedom
from idaes.core.solvers import get_solver
import pandas as pd

# Step 1 — Flowsheet
m = ConcreteModel()
m.fs = FlowsheetBlock(dynamic=False)

# Step 2 — Property packages
m.fs.prop_h2 = GenericParameterBlock(**get_prop({"H2O","H2"}, phases={"Vap"}, eos=EosType.IDEAL))
m.fs.prop_o2 = GenericParameterBlock(**get_prop({"O2","H2O"}, phases={"Vap"}, eos=EosType.IDEAL))

# Step 3 — SOEC unit
m.fs.soec = SoecDesign(
    oxygen_side_property_package=m.fs.prop_o2,
    hydrogen_side_property_package=m.fs.prop_h2,
    has_heat_transfer=True,
)
TransformationFactory("network.expand_arcs").apply_to(m)

# Step 4 — Fix all inputs (confirmed working from previous session)
m.fs.soec.hydrogen_side_inlet.flow_mol[0].fix(0.10)
m.fs.soec.hydrogen_side_inlet.mole_frac_comp[0,"H2O"].fix(0.90)
m.fs.soec.hydrogen_side_inlet.mole_frac_comp[0,"H2"].fix(0.10)
m.fs.soec.hydrogen_side_inlet.temperature[0].fix(1073.15)
m.fs.soec.hydrogen_side_inlet.pressure[0].fix(1.5e5)

m.fs.soec.oxygen_side_inlet.flow_mol[0].fix(0.10)
m.fs.soec.oxygen_side_inlet.mole_frac_comp[0,"O2"].fix(0.21)
m.fs.soec.oxygen_side_inlet.mole_frac_comp[0,"H2O"].fix(0.79)
m.fs.soec.oxygen_side_inlet.temperature[0].fix(1073.15)
m.fs.soec.oxygen_side_inlet.pressure[0].fix(1.5e5)

m.fs.soec.water_utilization[0].fix(0.80)
m.fs.soec.heat[0].fix(0.0)
m.fs.soec.cell_potential[0].fix(1.28)
m.fs.soec.sweep_heater.outlet.temperature[0].fix(1073.15)

print("DOF:", degrees_of_freedom(m))  # Must be 0

# Step 5 — Single-point validation solve
solver = get_solver()
try:
    m.fs.soec.initialize(outlvl=0)
except:
    pass

result = solver.solve(m, tee=False)
print("Status:", result.solver.termination_condition)
print("V_cell:", round(value(m.fs.soec.cell_potential[0]), 4), "V")
print("Current:", round(value(m.fs.soec.current[0]), 1), "A")
print("T_h2_out:", round(value(m.fs.soec.hydrogen_side_outlet_temperature[0]), 2), "K")
print("dT:", round(value(m.fs.soec.hydrogen_side_outlet_temperature[0]) -
                   value(m.fs.soec.oxygen_side_outlet_temperature[0]), 2), "K")
print("\n✅ Model rebuilt — ready for sweep")

DOF: 0
2026-03-28 03:17:55 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Starting initialization
2026-03-28 03:17:55 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Property initialization: optimal - Optimal Solution Found.
2026-03-28 03:17:55 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Starting initialization
2026-03-28 03:17:55 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property initialization: optimal - Optimal Solution Found.
2026-03-28 03:17:55 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property package initialization: optimal - Optimal Solution Found.
2026-03-28 03:17:55 [INFO] idaes.init.fs.soec.h2_inlet_translator: Initialization Complete optimal - Optimal Solution Found.
2026-03-28 03:17:55 [INFO] idaes.init.fs.soec.electrolysis_reactor.control_volume.properties_in: Starting initialization
that are not Var, Constraint, Objective, or the model.  Skipping.
2026-03-28 03:17:55 [INFO] idaes.init.fs

In [25]:
import pandas as pd
from pyomo.environ import value

results = []
V_targets = [0.95, 1.00, 1.05, 1.10, 1.15, 1.20, 1.25, 1.28]

for v_target in V_targets:
    m.fs.soec.cell_potential[0].fix(v_target)
    
    try:
        m.fs.soec.initialize(outlvl=0)
    except:
        pass
    
    res = solver.solve(m, tee=False)
    
    if str(res.solver.termination_condition) == "optimal":
        v_cell = value(m.fs.soec.cell_potential[0])
        T_h2   = value(m.fs.soec.hydrogen_side_outlet_temperature[0])
        T_o2   = value(m.fs.soec.oxygen_side_outlet_temperature[0])
        dT     = T_h2 - T_o2
        current = value(m.fs.soec.current[0])
        j      = current / (900 * 360)
        safe   = int(0.9 <= v_cell <= 1.3 and abs(dT) <= 20)
        
        results.append({
            "cell_potential_V": round(v_cell, 4),
            "current_A":        round(current, 2),
            "j_Acm2":           round(j, 6),
            "T_h2_out_K":       round(T_h2, 2),
            "T_o2_out_K":       round(T_o2, 2),
            "dT_K":             round(dT, 2),
            "safe":             safe
        })
        print(f"V={v_cell:.2f}V  j={j:.4f}  dT={dT:.1f}K  safe={safe}")
    else:
        print(f"WARNING: failed at V={v_target}")

df = pd.DataFrame(results)
df.to_csv("pinn_training_data.csv", index=False)
print("\n", df.to_string())

2026-03-28 03:18:06 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Starting initialization
2026-03-28 03:18:07 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Property initialization: optimal - Optimal Solution Found.
2026-03-28 03:18:07 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Starting initialization
2026-03-28 03:18:07 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property initialization: optimal - Optimal Solution Found.
2026-03-28 03:18:07 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property package initialization: optimal - Optimal Solution Found.
2026-03-28 03:18:07 [INFO] idaes.init.fs.soec.h2_inlet_translator: Initialization Complete optimal - Optimal Solution Found.
2026-03-28 03:18:07 [INFO] idaes.init.fs.soec.electrolysis_reactor.control_volume.properties_in: Starting initialization
that are not Var, Constraint, Objective, or the model.  Skipping.
2026-03-28 03:18:07 [INFO] idaes.init.fs.soec.e

In [26]:
import pandas as pd
from pyomo.environ import value

results = []

# Sweep water_utilization instead — this is the true independent variable
# cell_potential becomes the OUTPUT (what the physics dictates)
m.fs.soec.cell_potential[0].unfix()  # let voltage float
m.fs.soec.current[0].fix(13893.89)   # hold current from validated point

util_targets = [0.50, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

for util in util_targets:
    m.fs.soec.water_utilization[0].fix(util)
    
    try:
        m.fs.soec.initialize(outlvl=0)
    except:
        pass
    
    res = solver.solve(m, tee=False)
    
    if str(res.solver.termination_condition) == "optimal":
        v_cell  = value(m.fs.soec.cell_potential[0])
        T_h2    = value(m.fs.soec.hydrogen_side_outlet_temperature[0])
        T_o2    = value(m.fs.soec.oxygen_side_outlet_temperature[0])
        dT      = T_h2 - T_o2
        current = value(m.fs.soec.current[0])
        j       = current / (900 * 360)
        safe    = int(0.9 <= v_cell <= 1.3 and abs(dT) <= 20)
        
        results.append({
            "water_utilization":  util,
            "cell_potential_V":   round(v_cell, 4),
            "current_A":          round(current, 2),
            "j_Acm2":             round(j, 6),
            "T_h2_out_K":         round(T_h2, 2),
            "T_o2_out_K":         round(T_o2, 2),
            "dT_K":               round(dT, 2),
            "safe":               safe
        })
        print(f"util={util:.2f}  V={v_cell:.4f}V  dT={dT:.1f}K  safe={safe}")
    else:
        print(f"WARNING: failed at util={util}")

df = pd.DataFrame(results)
df.to_csv("pinn_training_data.csv", index=False)
print("\n", df.to_string())

2026-03-28 03:19:26 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Starting initialization
2026-03-28 03:19:27 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Property initialization: optimal - Optimal Solution Found.
2026-03-28 03:19:27 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Starting initialization
2026-03-28 03:19:27 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property initialization: optimal - Optimal Solution Found.
2026-03-28 03:19:27 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property package initialization: optimal - Optimal Solution Found.
2026-03-28 03:19:27 [INFO] idaes.init.fs.soec.h2_inlet_translator: Initialization Complete optimal - Optimal Solution Found.
2026-03-28 03:19:27 [INFO] idaes.init.fs.soec.electrolysis_reactor.control_volume.properties_in: Starting initialization
that are not Var, Constraint, Objective, or the model.  Skipping.
2026-03-28 03:19:27 [INFO] idaes.init.fs.soec.e

In [27]:
import pandas as pd
from pyomo.environ import value

# Reset to clean known-good state
m.fs.soec.current[0].unfix()
m.fs.soec.cell_potential[0].fix(1.28)
m.fs.soec.water_utilization[0].fix(0.80)

results = []

# Sweep flow_mol as the RE-surge proxy
# Higher flow = more steam = higher current = simulates RE power surge
flow_targets = [0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]

for flow in flow_targets:
    # Update both inlet flows together
    m.fs.soec.hydrogen_side_inlet.flow_mol[0].fix(flow)
    m.fs.soec.oxygen_side_inlet.flow_mol[0].fix(flow)

    try:
        m.fs.soec.initialize(outlvl=0)
    except:
        pass

    res = solver.solve(m, tee=False)

    if str(res.solver.termination_condition) == "optimal":
        v_cell  = value(m.fs.soec.cell_potential[0])
        T_h2    = value(m.fs.soec.hydrogen_side_outlet_temperature[0])
        T_o2    = value(m.fs.soec.oxygen_side_outlet_temperature[0])
        dT      = T_h2 - T_o2
        current = value(m.fs.soec.current[0])
        j       = current / (900 * 360)
        safe    = int(0.9 <= v_cell <= 1.3 and abs(dT) <= 20)

        results.append({
            "flow_mol":          flow,
            "cell_potential_V":  round(v_cell, 4),
            "current_A":         round(current, 2),
            "j_Acm2":            round(j, 6),
            "T_h2_out_K":        round(T_h2, 2),
            "T_o2_out_K":        round(T_o2, 2),
            "dT_K":              round(dT, 2),
            "safe":              safe
        })
        print(f"flow={flow:.2f}  V={v_cell:.4f}V  j={j:.5f}  dT={dT:.1f}K  safe={safe}")
    else:
        print(f"WARNING: failed at flow={flow}")

df = pd.DataFrame(results)
df.to_csv("pinn_training_data.csv", index=False)
print("\n", df.to_string())

2026-03-28 03:20:06 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Starting initialization
2026-03-28 03:20:07 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Property initialization: optimal - Optimal Solution Found.
2026-03-28 03:20:07 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Starting initialization
2026-03-28 03:20:08 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property initialization: optimal - Optimal Solution Found.
2026-03-28 03:20:08 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property package initialization: optimal - Optimal Solution Found.
2026-03-28 03:20:08 [INFO] idaes.init.fs.soec.h2_inlet_translator: Initialization Complete optimal - Optimal Solution Found.
2026-03-28 03:20:08 [INFO] idaes.init.fs.soec.electrolysis_reactor.control_volume.properties_in: Starting initialization
that are not Var, Constraint, Objective, or the model.  Skipping.
2026-03-28 03:20:08 [INFO] idaes.init.fs.soec.e

In [28]:
# Unfix sweep_heater outlet — let it be determined by energy balance
# Instead fix the sweep_heater heat_duty = 0 (no external heating)
m.fs.soec.sweep_heater.outlet.temperature[0].unfix()
m.fs.soec.sweep_heater.control_volume.heat[0].fix(0.0)  # adiabatic sweep heater

from idaes.core.util.model_statistics import degrees_of_freedom
print("DOF:", degrees_of_freedom(m))  # Must still be 0

DOF: 0


In [29]:
results = []

V_targets   = [1.10, 1.15, 1.20, 1.25, 1.28]
flow_targets = [0.05, 0.10, 0.20, 0.30, 0.50]

for v in V_targets:
    m.fs.soec.cell_potential[0].fix(v)
    for flow in flow_targets:
        m.fs.soec.hydrogen_side_inlet.flow_mol[0].fix(flow)
        m.fs.soec.oxygen_side_inlet.flow_mol[0].fix(flow)

        try:
            m.fs.soec.initialize(outlvl=0)
        except:
            pass

        res = solver.solve(m, tee=False)

        if str(res.solver.termination_condition) == "optimal":
            v_cell  = value(m.fs.soec.cell_potential[0])
            T_h2    = value(m.fs.soec.hydrogen_side_outlet_temperature[0])
            T_o2    = value(m.fs.soec.oxygen_side_outlet_temperature[0])
            dT      = T_h2 - T_o2
            current = value(m.fs.soec.current[0])
            j       = current / (900 * 360)
            safe    = int(0.9 <= v_cell <= 1.3 and abs(dT) <= 20)

            results.append({
                "flow_mol":         flow,
                "cell_potential_V": round(v_cell, 4),
                "current_A":        round(current, 2),
                "j_Acm2":           round(j, 6),
                "T_h2_out_K":       round(T_h2, 2),
                "T_o2_out_K":       round(T_o2, 2),
                "dT_K":             round(dT, 2),
                "safe":             safe
            })
            print(f"V={v:.2f} flow={flow:.2f} j={j:.4f} dT={dT:.1f}K safe={safe}")
        else:
            print(f"FAILED: V={v} flow={flow}")

df = pd.DataFrame(results)
df.to_csv("pinn_training_data.csv", index=False)
print(f"\n{len(df)} data points saved")
print(df[["cell_potential_V","flow_mol","j_Acm2","dT_K","safe"]].to_string())

2026-03-28 03:21:20 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Starting initialization
2026-03-28 03:21:20 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Property initialization: optimal - Optimal Solution Found.
2026-03-28 03:21:20 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Starting initialization
2026-03-28 03:21:20 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property initialization: optimal - Optimal Solution Found.
2026-03-28 03:21:20 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property package initialization: optimal - Optimal Solution Found.
2026-03-28 03:21:20 [INFO] idaes.init.fs.soec.h2_inlet_translator: Initialization Complete optimal - Optimal Solution Found.
2026-03-28 03:21:20 [INFO] idaes.init.fs.soec.electrolysis_reactor.control_volume.properties_in: Starting initialization
that are not Var, Constraint, Objective, or the model.  Skipping.
2026-03-28 03:21:20 [INFO] idaes.init.fs.soec.e

In [30]:
import shutil
shutil.copy("pinn_training_data.csv", "pinn_training_data_v1_25pts.csv")
print("CSV backed up ✅")
print(df[df['safe']==1][['cell_potential_V','dT_K','j_Acm2']].to_string())

CSV backed up ✅
    cell_potential_V   dT_K    j_Acm2
20              1.28 -16.06  0.021441
21              1.28 -16.06  0.042882
22              1.28 -16.06  0.085765
23              1.28 -16.06  0.128647
24              1.28 -16.06  0.214412


In [31]:
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import pickle

# Load training data
df = pd.read_csv("pinn_training_data.csv")

# Features: cell_potential and dT (the two observable state variables)
X = df[["cell_potential_V", "dT_K"]].values
y = df["safe"].values

print("Training data shape:", X.shape)
print("Safe points:", y.sum(), "/ Unsafe:", (1-y).sum())

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train PINN boundary classifier
pinn = MLPClassifier(
    hidden_layer_sizes=(16, 16),
    activation='relu',
    max_iter=1000,
    random_state=42
)
pinn.fit(X_scaled, y)

# Evaluate
y_pred = pinn.predict(X_scaled)
print("\n=== PINN Classifier Report ===")
print(classification_report(y, y_pred, target_names=["Unsafe", "Safe"]))

# Save the trained PINN + scaler
with open("pinn_boundary_classifier.pkl", "wb") as f:
    pickle.dump({"model": pinn, "scaler": scaler}, f)
print("PINN saved to pinn_boundary_classifier.pkl ✅")

ModuleNotFoundError: No module named 'sklearn'

In [32]:
# ============================================================
# FULL REBUILD — run this first
# ============================================================
from pyomo.environ import ConcreteModel, value, TransformationFactory
from idaes.core import FlowsheetBlock
from idaes.models_extra.power_generation.unit_models.soec_design import SoecDesign
from idaes.models_extra.power_generation.properties.natural_gas_PR import get_prop, EosType
from idaes.models.properties.modular_properties.base.generic_property import GenericParameterBlock
from idaes.core.util.model_statistics import degrees_of_freedom
from idaes.core.solvers import get_solver

m = ConcreteModel()
m.fs = FlowsheetBlock(dynamic=False)
m.fs.prop_h2 = GenericParameterBlock(**get_prop({"H2O","H2"}, phases={"Vap"}, eos=EosType.IDEAL))
m.fs.prop_o2 = GenericParameterBlock(**get_prop({"O2","H2O"}, phases={"Vap"}, eos=EosType.IDEAL))
m.fs.soec = SoecDesign(
    oxygen_side_property_package=m.fs.prop_o2,
    hydrogen_side_property_package=m.fs.prop_h2,
    has_heat_transfer=True,
)
TransformationFactory("network.expand_arcs").apply_to(m)

m.fs.soec.hydrogen_side_inlet.flow_mol[0].fix(0.10)
m.fs.soec.hydrogen_side_inlet.mole_frac_comp[0,"H2O"].fix(0.90)
m.fs.soec.hydrogen_side_inlet.mole_frac_comp[0,"H2"].fix(0.10)
m.fs.soec.hydrogen_side_inlet.temperature[0].fix(1073.15)
m.fs.soec.hydrogen_side_inlet.pressure[0].fix(1.5e5)
m.fs.soec.oxygen_side_inlet.flow_mol[0].fix(0.10)
m.fs.soec.oxygen_side_inlet.mole_frac_comp[0,"O2"].fix(0.21)
m.fs.soec.oxygen_side_inlet.mole_frac_comp[0,"H2O"].fix(0.79)
m.fs.soec.oxygen_side_inlet.temperature[0].fix(1073.15)
m.fs.soec.oxygen_side_inlet.pressure[0].fix(1.5e5)
m.fs.soec.water_utilization[0].fix(0.80)
m.fs.soec.heat[0].fix(0.0)
m.fs.soec.cell_potential[0].fix(1.28)
m.fs.soec.sweep_heater.outlet.temperature[0].unfix()
m.fs.soec.sweep_heater.control_volume.heat[0].fix(0.0)

solver = get_solver()
try:
    m.fs.soec.initialize(outlvl=0)
except:
    pass
result = solver.solve(m, tee=False)

print("DOF:", degrees_of_freedom(m))
print("Status:", result.solver.termination_condition)
print("Model ready ✅")

2026-03-28 04:30:01 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Starting initialization
2026-03-28 04:30:01 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Property initialization: optimal - Optimal Solution Found.
2026-03-28 04:30:01 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Starting initialization
2026-03-28 04:30:01 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property initialization: optimal - Optimal Solution Found.
2026-03-28 04:30:01 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property package initialization: optimal - Optimal Solution Found.
2026-03-28 04:30:02 [INFO] idaes.init.fs.soec.h2_inlet_translator: Initialization Complete optimal - Optimal Solution Found.
2026-03-28 04:30:02 [INFO] idaes.init.fs.soec.electrolysis_reactor.control_volume.properties_in: Starting initialization
that are not Var, Constraint, Objective, or the model.  Skipping.
2026-03-28 04:30:02 [INFO] idaes.init.fs.soec.e

In [33]:
import gymnasium as gym
import numpy as np
import pickle
from pyomo.environ import value

class SOECAssetHealthEnv(gym.Env):
    """
    Single-objective SOEC asset health environment.
    State:  [cell_potential_V, dT_K, flow_mol_normalized]
    Action: delta_flow in [-0.05, +0.05] mol/s per step
    Reward: shaped for asset health only
    """
    def __init__(self, idaes_model, solver, pinn_path="pinn_boundary_classifier.pkl"):
        super().__init__()
        self.m = idaes_model
        self.solver = solver
        self.flow_min, self.flow_max = 0.05, 0.50
        self.current_flow = 0.10

        # Load PINN classifier
        with open(pinn_path, "rb") as f:
            pinn_data = pickle.load(f)
        self.pinn = pinn_data["model"]
        self.pinn_scaler = pinn_data["scaler"]

        # Action: change flow by ±0.05 mol/s
        self.action_space = gym.spaces.Box(
            low=np.array([-0.05]), high=np.array([0.05]), dtype=np.float32
        )
        # State: [V_cell, dT, flow_normalized]
        self.observation_space = gym.spaces.Box(
            low=np.array([0.9, -700.0, 0.0]),
            high=np.array([1.4,  100.0, 1.0]),
            dtype=np.float32
        )

    def _get_obs(self):
        v = value(self.m.fs.soec.cell_potential[0])
        T_h2 = value(self.m.fs.soec.hydrogen_side_outlet_temperature[0])
        T_o2 = value(self.m.fs.soec.oxygen_side_outlet_temperature[0])
        dT = T_h2 - T_o2
        flow_norm = (self.current_flow - self.flow_min) / (self.flow_max - self.flow_min)
        return np.array([v, dT, flow_norm], dtype=np.float32)

    def _compute_reward(self, v_cell, dT, prev_flow, new_flow):
        reward = 0.0
        if 0.9 <= v_cell <= 1.3 and abs(dT) <= 20:
            reward += 1.0   # in safe window
        else:
            if not (0.9 <= v_cell <= 1.3):
                reward -= 10.0 * abs(v_cell - 1.1) / 0.2
            if abs(dT) > 20:
                reward -= 50.0 * (abs(dT) - 20.0) / 20.0
        # Penalise rapid load changes
        reward -= 2.0 * abs(new_flow - prev_flow) / 0.05
        return float(reward)

    def step(self, action):
        prev_flow = self.current_flow
        delta = float(np.clip(action[0], -0.05, 0.05))
        self.current_flow = float(np.clip(self.current_flow + delta,
                                          self.flow_min, self.flow_max))

        # Apply to IDAES model
        self.m.fs.soec.hydrogen_side_inlet.flow_mol[0].fix(self.current_flow)
        self.m.fs.soec.oxygen_side_inlet.flow_mol[0].fix(self.current_flow)

        try:
            self.m.fs.soec.initialize(outlvl=0)
        except:
            pass
        result = self.solver.solve(self.m, tee=False)

        obs = self._get_obs()
        v_cell, dT = obs[0], obs[1]
        reward = self._compute_reward(v_cell, dT, prev_flow, self.current_flow)

        # PINN safety check
        X = self.pinn_scaler.transform([[v_cell, dT]])
        pinn_safe = bool(self.pinn.predict(X)[0])

        terminated = False
        truncated = False
        info = {"pinn_safe": pinn_safe, "v_cell": v_cell, "dT": dT,
                "flow": self.current_flow, "solver_ok":
                str(result.solver.termination_condition) == "optimal"}
        return obs, reward, terminated, truncated, info

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_flow = 0.10
        self.m.fs.soec.hydrogen_side_inlet.flow_mol[0].fix(0.10)
        self.m.fs.soec.oxygen_side_inlet.flow_mol[0].fix(0.10)
        try:
            self.m.fs.soec.initialize(outlvl=0)
        except:
            pass
        self.solver.solve(self.m, tee=False)
        return self._get_obs(), {}

# Test the environment
env = SOECAssetHealthEnv(m, solver)
obs, info = env.reset()
print("Reset obs:", obs)

# Take 3 test steps
for i in range(3):
    action = env.action_space.sample()
    obs, reward, term, trunc, info = env.step(action)
    print(f"Step {i+1}: flow={info['flow']:.2f} V={info['v_cell']:.4f} "
          f"dT={info['dT']:.1f} reward={reward:.2f} safe={info['pinn_safe']}")

print("\nGymnasium environment verified ✅")

2026-03-28 04:31:34 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Starting initialization


C:\Users\LOGIN\miniconda3\envs\idaes-env\lib\site-packages\gymnasium\spaces\box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
C:\Users\LOGIN\miniconda3\envs\idaes-env\lib\site-packages\gymnasium\spaces\box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


2026-03-28 04:31:34 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_in: Property initialization: optimal - Optimal Solution Found.
2026-03-28 04:31:34 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Starting initialization
2026-03-28 04:31:34 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property initialization: optimal - Optimal Solution Found.
2026-03-28 04:31:34 [INFO] idaes.init.fs.soec.h2_inlet_translator.properties_out: Property package initialization: optimal - Optimal Solution Found.
2026-03-28 04:31:34 [INFO] idaes.init.fs.soec.h2_inlet_translator: Initialization Complete optimal - Optimal Solution Found.
2026-03-28 04:31:34 [INFO] idaes.init.fs.soec.electrolysis_reactor.control_volume.properties_in: Starting initialization
that are not Var, Constraint, Objective, or the model.  Skipping.
2026-03-28 04:31:34 [INFO] idaes.init.fs.soec.electrolysis_reactor.control_volume.properties_in: Property initialization: optimal - Optimal Solution Fou